# Loading Data 

In [ ]:
using Pkg
using CSV, DataFrames
using Statistics
using GLM
using StatsModels
using Plots
using Dates
using ShiftedArrays, CovarianceMatrices


In [1]:
cd(@__DIR__)
pwd()

"/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data"

In [ ]:
# Load the data and final cleaning
df = CSV.read("romer-romer_data.csv", DataFrame);
select!(df, Not(1));
df.quarter = Date.(replace.(df.quarter, "Q1"=>"01", "Q2"=>"04", "Q3"=>"07", "Q4"=>"10"), dateformat"yyyymm");


Row,index,quarter,FR007,cpi,ngdp_growth,rgdp_growth,target_gdp,target_cpi,gdp_gap,cpi_gap
,Int64,String7,Float64,Float64,Float64?,Float64?,Float64,Float64,Float64?,Float64
1,0,2000Q1,2.50631,0.100897,missing,missing,7.0,2.0,missing,-1.8991
2,1,2000Q2,2.39815,0.0969205,missing,missing,7.0,2.0,missing,-1.90308
3,2,2000Q3,2.33258,0.264006,missing,missing,7.0,2.0,missing,-1.73599
4,3,2000Q4,2.3609,0.930059,missing,missing,7.0,2.0,missing,-1.06994
5,4,2001Q1,2.50195,0.66374,7.78938,4.79272,7.0,1.5,0.789375,-0.83626
6,5,2001Q2,2.38814,1.56724,8.70938,6.29685,7.0,1.5,1.70938,0.067239
7,6,2001Q3,2.28799,0.792621,9.73152,7.6181,7.0,1.5,2.73152,-0.707379
8,7,2001Q4,2.1527,-0.133722,9.93129,8.66243,7.0,1.5,2.93129,-1.63372
9,8,2002Q1,2.05543,-0.586898,10.0335,9.85281,7.0,1.5,3.03353,-2.0869


In [ ]:
# Create new variables for positive and negative GDP gaps
df.not_meet_target = df.gdp_gap .< 0;
df.gap_pos = df.gdp_gap .* (1 .- df.not_meet_target);
df.gap_neg = df.gdp_gap .* df.not_meet_target;


96-element Vector{Union{Missing, Float64}}:
   missing
   missing
   missing
   missing
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  ⋮
  0.0
  0.0
 -0.413013207983532
 -0.029718913117832813
 -3.722233398108405
 -0.6119852643141588
 -0.3774418843062577
 -1.0836295893040315
  0.0

In [ ]:
# Create lagged variables for the regression
df.FR007_l1   = lag(df.FR007, 1);
df.cpi_gap_l1 = lag(df.cpi_gap, 1);
df.gap_pos_l1 = lag(df.gap_pos, 1);
df.gap_neg_l1 = lag(df.gap_neg, 1);

96-element ShiftedArrays.ShiftedVector{Union{Missing, Float64}, Missing, Vector{Union{Missing, Float64}}}:
   missing
   missing
   missing
   missing
   missing
  0.0
  0.0
  0.0
  0.0
  0.0
  ⋮
  0.0
  0.0
  0.0
 -0.413013207983532
 -0.029718913117832813
 -3.722233398108405
 -0.6119852643141588
 -0.3774418843062577
 -1.0836295893040315

In [15]:
# Drop missing values
df_reg = dropmissing(df, [:FR007, :FR007_l1, :cpi_gap_l1, :gap_pos_l1, :gap_neg_l1])


Row,quarter,FR007,cpi,ngdp_growth,rgdp_growth,target_gdp,target_cpi,gdp_gap,cpi_gap,not_meet_target,gap_pos,gap_neg,FR007_l1,cpi_gap_l1,gap_pos_l1,gap_neg_l1
,Date,Float64,Float64,Float64?,Float64?,Float64,Float64,Float64?,Float64,Bool?,Float64?,Float64?,Float64,Float64,Float64,Float64
1,2001-04-01,2.38814,1.56724,8.70938,6.29685,7.0,1.5,1.70938,0.067239,false,1.70938,0.0,2.50195,-0.83626,0.789375,0.0
2,2001-07-01,2.28799,0.792621,9.73152,7.6181,7.0,1.5,2.73152,-0.707379,false,2.73152,0.0,2.38814,0.067239,1.70938,0.0
3,2001-10-01,2.1527,-0.133722,9.93129,8.66243,7.0,1.5,2.93129,-1.63372,false,2.93129,0.0,2.28799,-0.707379,2.73152,0.0
4,2002-01-01,2.05543,-0.586898,10.0335,9.85281,7.0,1.5,3.03353,-2.0869,false,3.03353,0.0,2.1527,-1.63372,2.93129,0.0
5,2002-04-01,1.9529,-0.991354,11.3221,10.8678,7.0,1.5,4.32211,-2.49135,false,4.32211,0.0,2.05543,-2.0869,3.03353,0.0
6,2002-07-01,2.03149,-0.757691,12.117,11.4102,7.0,1.5,5.11696,-2.25769,false,5.11696,0.0,1.9529,-2.49135,4.32211,0.0
7,2002-10-01,2.26109,-0.593172,12.3935,11.1035,7.0,1.5,5.39347,-2.09317,false,5.39347,0.0,2.03149,-2.25769,5.11696,0.0
8,2003-01-01,2.18534,0.469622,13.3226,11.0135,7.0,1.0,6.32264,-0.530378,false,6.32264,0.0,2.26109,-2.09317,5.39347,0.0
9,2003-04-01,2.04885,0.609439,12.2097,9.79562,7.0,1.0,5.2097,-0.390561,false,5.2097,0.0,2.18534,-0.530378,6.32264,0.0
